# PERF-COMPOSITION. Strategy Interaction and Bottleneck Shift | 策略相互作用与瓶颈转移

**难度：** Hard | **环境：** CPU 机制练习 + 单卡 GPU 机制实验  
**标签：** `策略组合` `依赖冲突` `边际收益` `瓶颈转移` | **目标人群：** 端到端性能工程学习者

---

## 本节导读

通过单项准入的候选组合后不一定保留各自收益。候选可能依赖同一执行顺序、争用同一资源，或在缓解原约束后暴露新的主约束。

本节先检查候选依赖与冲突，再用同一基线比较单项和组合实测结果，最后根据边际收益、守护条件和新瓶颈选择保留或回退。


## 前置阅读

- [Part 02 · PERF-CANDIDATE](./PERF-CANDIDATE_Constraint_to_Candidate_and_Cost_Transfer.ipynb)：建立单候选作用对象与代价契约。
- [Part 02 · PERF-REGRESSION](./PERF-REGRESSION_Paired_Benchmark_and_Regression_Thresholds.ipynb)：理解配对复测与性能回归阈值。
- [Part 02 · PERF-TRACE](./PERF-TRACE_Trace_Dependencies_and_Critical_Path.ipynb)：在组合后重新定位关键路径。
---


### Step 1：组合前先检查依赖、冲突与顺序

候选之间可能存在前置依赖、格式冲突和资源冲突。依赖要求某候选先于另一候选生效；冲突表示两个候选不能同时成立；执行顺序决定最终数据类型、布局、缓存状态或计算路径。

| 关系 | 示例 | 组合动作 |
| --- | --- | --- |
| 前置依赖 | 执行后端必须先支持目标产物格式 | 依赖候选排在消费候选之前 |
| 格式冲突 | 两个候选要求不同布局或数据类型 | 拒绝组合或选择一个候选 |
| 资源冲突 | 两个候选同时增加工作区或通信流 | 记录组合峰值与暴露等待 |
| 顺序敏感 | 先融合再量化与先量化再融合命中不同路径 | 将顺序写入组合身份 |

![策略组合、复测与瓶颈转移](../docs/public/02_PyTorch_Algorithms/perf_composition_flow.svg)


### Step 2：组合收益来自实测边际，而不是单项收益相加

组合对照至少包含同一基线、候选 A、候选 B 和 A+B。组合相对最佳单项的边际变化回答第二个候选是否仍有价值；组合相对基线的变化回答整体是否仍满足优化目标。

| 对照 | 回答的问题 |
| --- | --- |
| 基线 vs A / B | 每个候选能否独立生效 |
| 最佳单项 vs A+B | 第二个候选是否保留边际收益 |
| 基线 vs A+B | 组合是否达到目标且通过守护条件 |
| A+B vs 回退方案 | 哪个候选应在失败时撤销 |


### Step 3：每次组合后都要重新确认主约束

候选 A 缓解计算后，搬运可能成为最长阶段；候选 B 再减少搬运后，启动、排队或通信可能成为新主约束。因此组合不能沿用单项实验中的瓶颈标签。

| 组合结果 | 决策 |
| --- | --- |
| 通过守护条件，且优于最佳单项 | 保留组合并记录新主约束 |
| 优于基线，但不优于最佳单项 | 回退没有边际收益的候选 |
| 守护条件失败或不优于基线 | 回退到通过准入的单项或基线 |

![候选相互作用与瓶颈转移示例](../docs/public/02_PyTorch_Algorithms/perf_composition_bottleneck_shift.svg)


### Step 4：实现组合顺序、边际收益与回退判断

题目区先验证候选顺序是否满足依赖且不存在冲突，再使用真实测量值计算组合相对基线和最佳单项的变化，最后检查守护条件并选择回退目标。

| TODO / 函数 | 学习者完成的机制 | 必须满足的约束 | 测试证据 |
| --- | --- | --- | --- |
| TODO 1 / `validate_order` | 检查依赖、冲突和重复候选 | 依赖必须先出现；冲突不得共存 | 合法顺序、缺失依赖与冲突 |
| TODO 2 / `measured_margins` | 比较基线、单项和组合目标指标 | 不把单项收益直接相加 | 协同、收益抵消与无增益 |
| TODO 3 / `decide_composition` | 检查守护条件与最佳已通过方案 | 硬门失败直接回退 | 保留组合、保留单项与恢复基线 |


In [ ]:
from collections.abc import Sequence


In [ ]:
def validate_order(order: Sequence[str], specs: dict[str, dict[str, set[str]]]) -> list[str]:
    """验证候选顺序中的依赖、冲突和重复项。"""
    # ==========================================
    # TODO 1：验证组合顺序
    # ==========================================
    # seen = ???  # 已按顺序通过检查的候选集合
    # validated = ???  # 保留原顺序的候选列表
    raise NotImplementedError("TODO 1：请验证组合顺序")
    for name in order:
        if name not in specs or name in seen:
            raise ValueError("候选未知或重复")
        # requires = ???  # 当前候选的前置依赖集合
        # conflicts = ???  # 当前候选主动声明的冲突集合
        # reverse_conflicts = ???  # 已加入候选对当前候选声明的冲突
        if not requires <= seen:
            raise ValueError(f"候选 {name} 的依赖尚未满足")
        if conflicts & seen or reverse_conflicts:
            raise ValueError(f"候选 {name} 与已有候选冲突")
        seen.add(name)
        validated.append(name)
    return validated


def measured_margins(baseline: float, singles: dict[str, float], combined: float) -> dict[str, float | str]:
    """计算越低越好的目标指标上的基线收益和组合边际收益。"""
    if baseline <= 0 or combined <= 0 or not singles or any(value <= 0 for value in singles.values()):
        raise ValueError("测量值必须为正，且至少提供一个单项结果")
    # ==========================================
    # TODO 2：计算实测边际收益
    # ==========================================
    # best_single_name = ???  # 延迟最低的单项候选
    # best_single_value = ???  # 最佳单项的实测值
    # gain_vs_baseline = ???  # 组合相对基线的下降比例
    # gain_vs_best_single = ???  # 组合相对最佳单项的下降比例
    raise NotImplementedError("TODO 2：请计算实测边际收益")
    return {"best_single": best_single_name, "gain_vs_baseline": gain_vs_baseline, "gain_vs_best_single": gain_vs_best_single}


def decide_composition(baseline: float, singles: dict[str, float], combined: float, guardrails_passed: bool, minimum_marginal_gain: float) -> dict[str, object]:
    """按守护条件和实测边际收益选择组合或回退目标。"""
    margins = measured_margins(baseline, singles, combined)
    # ==========================================
    # TODO 3：选择保留或回退
    # ==========================================
    # best_single_value = ???  # 最佳单项的实测值
    # best_single_passed = ???  # 最佳单项是否优于基线
    # combination_failed = ???  # 组合守护条件失败或未优于基线
    raise NotImplementedError("TODO 3：请完成组合决策")
    if combination_failed:
        if best_single_passed:
            decision, rollback_to = "keep_single", margins["best_single"]
        else:
            decision, rollback_to = "rollback_baseline", "baseline"
    elif margins["gain_vs_best_single"] >= minimum_marginal_gain:
        decision, rollback_to = "keep_combination", margins["best_single"]
    else:
        decision, rollback_to = "keep_single", margins["best_single"]
    return {**margins, "decision": decision, "rollback_to": rollback_to}


### 测试

运行下方测试，检查候选依赖、冲突、实测边际收益和三类回退结果。


In [ ]:
def _specs():
    """返回包含依赖和冲突的候选规格。"""
    return {"format": {"requires": set(), "conflicts": set()}, "kernel": {"requires": {"format"}, "conflicts": {"eager_only"}}, "eager_only": {"requires": set(), "conflicts": {"kernel"}}}


def test_order_dependencies_and_conflicts():
    """依赖必须先出现，冲突候选不能进入同一组合。"""
    assert validate_order(["format", "kernel"], _specs()) == ["format", "kernel"]
    asymmetric = _specs()
    asymmetric["eager_only"]["conflicts"] = set()
    for invalid, specs in ((["kernel", "format"], _specs()), (["format", "kernel", "eager_only"], _specs()), (["format", "kernel", "eager_only"], asymmetric)):
        try:
            validate_order(invalid, specs)
        except ValueError:
            pass
        else:
            raise AssertionError("缺失依赖或候选冲突必须报错")


def test_measured_margins_are_not_additive():
    """组合边际来自实测组合结果，而不是单项收益之和。"""
    margins = measured_margins(100.0, {"a": 80.0, "b": 85.0}, 76.0)
    assert margins["best_single"] == "a"
    assert round(margins["gain_vs_baseline"], 4) == 0.24
    assert round(margins["gain_vs_best_single"], 4) == 0.05


def test_keep_combination_single_and_baseline():
    """边际收益和守护条件决定保留组合、单项或基线。"""
    singles = {"a": 80.0, "b": 85.0}
    assert decide_composition(100.0, singles, 72.0, True, 0.05)["decision"] == "keep_combination"
    assert decide_composition(100.0, singles, 79.0, True, 0.05)["rollback_to"] == "a"
    failed_guardrail = decide_composition(100.0, singles, 72.0, False, 0.05)
    assert failed_guardrail["decision"] == "keep_single" and failed_guardrail["rollback_to"] == "a"
    assert decide_composition(100.0, {"a": 105.0}, 110.0, False, 0.05)["decision"] == "rollback_baseline"


test_order_dependencies_and_conflicts()
test_measured_margins_are_not_additive()
test_keep_combination_single_and_baseline()
print("All tests passed!")


---

🛑 **STOP HERE** 🛑
<br><br><br><br><br><br><br><br><br><br>
> 请先尝试自己完成代码并跑通测试。<br>
> 如果遇到困难没有思路，可以向下滚动查看参考答案。
<br><br><br><br><br><br><br><br><br><br>

---


## 参考代码与解析

### 代码


In [ ]:
def validate_order(order: Sequence[str], specs: dict[str, dict[str, set[str]]]) -> list[str]:
    """验证候选顺序中的依赖、冲突和重复项。"""
    # TODO 1：验证组合顺序。
    seen, validated = set(), []
    for name in order:
        if name not in specs or name in seen:
            raise ValueError("候选未知或重复")
        requires = set(specs[name].get("requires", set()))
        conflicts = set(specs[name].get("conflicts", set()))
        if not requires <= seen:
            raise ValueError(f"候选 {name} 的依赖尚未满足")
        reverse_conflicts = {prior for prior in seen if name in set(specs[prior].get("conflicts", set()))}
        if conflicts & seen or reverse_conflicts:
            raise ValueError(f"候选 {name} 与已有候选冲突")
        seen.add(name)
        validated.append(name)
    return validated


def measured_margins(baseline: float, singles: dict[str, float], combined: float) -> dict[str, float | str]:
    """计算越低越好的目标指标上的基线收益和组合边际收益。"""
    if baseline <= 0 or combined <= 0 or not singles or any(value <= 0 for value in singles.values()):
        raise ValueError("测量值必须为正，且至少提供一个单项结果")
    # TODO 2：计算实测边际收益。
    best_single_name = min(singles, key=singles.get)
    best_single_value = singles[best_single_name]
    gain_vs_baseline = (baseline - combined) / baseline
    gain_vs_best_single = (best_single_value - combined) / best_single_value
    return {"best_single": best_single_name, "gain_vs_baseline": gain_vs_baseline, "gain_vs_best_single": gain_vs_best_single}


def decide_composition(baseline: float, singles: dict[str, float], combined: float, guardrails_passed: bool, minimum_marginal_gain: float) -> dict[str, object]:
    """按守护条件和实测边际收益选择组合或回退目标。"""
    margins = measured_margins(baseline, singles, combined)
    # TODO 3：选择保留或回退。
    best_single_value = singles[margins["best_single"]]
    best_single_passed = best_single_value < baseline
    combination_failed = not guardrails_passed or margins["gain_vs_baseline"] <= 0
    if combination_failed:
        if best_single_passed:
            decision, rollback_to = "keep_single", margins["best_single"]
        else:
            decision, rollback_to = "rollback_baseline", "baseline"
    elif margins["gain_vs_best_single"] >= minimum_marginal_gain:
        decision, rollback_to = "keep_combination", margins["best_single"]
    else:
        decision, rollback_to = "keep_single", margins["best_single"]
    return {**margins, "decision": decision, "rollback_to": rollback_to}


### 解析

#### TODO 1：验证组合顺序

- 每个候选加入组合前检查前置依赖和双向冲突声明，组合顺序因此成为实验身份的一部分。
- 测试分别覆盖合法顺序、依赖倒置、对称冲突和单向声明的冲突。

#### TODO 2：计算实测边际收益

- 最佳单项来自真实测量的最低目标指标；组合边际以该结果为分母。
- 组合结果不从 A、B 的单项收益推算，测试使用非加和结果验证这一点。

#### TODO 3：选择保留或回退

- 组合守护条件失败时先回退到已通过且优于基线的最佳单项；没有可用单项时才恢复基线。
- 组合通过时仍保存最佳单项作为回退目标，并在下一轮重新确认主约束。


### Step 5：GPU 机制实验——单项、组合与瓶颈转移

CPU 题目确认依赖、冲突、边际收益和回退顺序；GPU 实验在同一流水线上同时测量基线、BF16、双缓冲重叠和 BF16+重叠，避免用两个单项收益相加推断组合结果。

| 学习问题 | CPU 已确认 | GPU 要确认与后续使用 |
| --- | --- | --- |
| 两个候选组合后是否仍有边际收益，主导阶段是否改变 | 组合顺序、实测边际和回退目标 | 基线/A/B/A+B 的逐负载中位数与 P95、输出质量、峰值显存、搬运/计算服务时间和新主导阶段 |

#### 5.1 环境与固定工作负载

本实验使用 `PERF-CUDA-MECHANISM`。首次运行按[性能分析实验 Quickstart](../topic_discussion/performance_analysis/quickstart.md#perf-cuda-mechanism-quickstart)创建并预检环境，实验身份与结果路径登记在[性能优化实验资产表](../topic_discussion/performance_optimization/experiment_assets.md)。本页使用固定合成张量，不下载模型或数据集。

| 配置层 | 本节默认口径 | 修改规则 |
| --- | --- | --- |
| 运行环境 | 单张 CUDA GPU；`PERF-CUDA-MECHANISM` | Profile 预检未返回 `status=ok` 时停止 |
| 工作负载 | 与 PERF-CANDIDATE 相同的 SwiGLU MLP、权重分布、负载矩阵和流水线批次数 | 修改任一工作负载字段后使用新标签，不跨协议拼接结果 |
| 对照矩阵 | FP32 串行 / BF16 串行 / FP32 重叠 / BF16 重叠 | A+B 必须在同一运行中实测，不从 A、B 推算 |
| 测量协议 | 12 轮轮换预热；24 轮逐轮轮换四条路径的执行顺序 | 每轮保留各路径样本，避免 GPU 状态漂移固定偏向后测路径 |
| 决策门槛 | 组合余弦相似度不低于 0.999；相对最佳单项至少再改善 3%；P95 不得回归超过 5% | 组合失败时回退到已通过的最佳单项，否则回到基线 |
| 再归因 | 比较组合路径的 H2D 与计算服务时间 | 该字段只表示阶段服务时间主导项，完整关键路径仍需 PERF-TRACE 复核 |
| 结果资产 | `benchmarks/results/perf_composition/<semantic_id>_<run_label>.json` | 正式复测使用唯一标签 |


In [1]:
RUN_MODE = "real_gpu"  # 默认值 dry_run；改为 real_gpu 执行真实组合矩阵。
RUN_LABEL = "rtx5070ti_composition_rotating_v2"  # 默认复测标签；更换硬件或协议后改为新的唯一标签。
PROFILE_ID = "PERF-CUDA-MECHANISM"  # 默认性能机制环境。
SEMANTIC_ID = "PERF-COMPOSITION"  # 稳定机制 ID。
WORKLOAD_ID = "perf-decoder-mlp-composition-matrix"  # 稳定工作负载 ID。
WORKLOAD_VERSION = "v2"  # v2 使用逐轮轮换顺序，减少四条路径的先后偏差。
GPU_CONFIG = {
    "hidden_size": 2048,          # 默认值 2048；与候选页保持一致。
    "intermediate_size": 5504,   # 默认值 5504；与候选页保持一致。
    "token_counts": [32, 256],   # 默认两档 Token 负载；逐负载比较边际收益。
    "pipeline_batches": 8,       # 默认值 8；提供稳定的重叠窗口。
    "warmup_steps": 12,          # 默认值 12；四条路径轮换预热且不计入统计。
    "repeat_steps": 24,          # 默认值 24；每轮轮换路径顺序并统计中位数与 P95。
    "minimum_marginal_gain": 0.03, # 默认值 3%；组合相对最佳单项的最低边际收益。
    "minimum_cosine": 0.999,     # 默认值 0.999；组合输出质量门槛。
    "maximum_p95_ratio": 1.05,   # 默认值 1.05；组合 P95 最多允许比最佳单项回归 5%。
    "seed": 42,                  # 默认值 42；固定权重与输入。
}
RESULTS_RELATIVE_DIR = "benchmarks/results/perf_composition"


#### 5.2 执行组合矩阵并保存 JSON

下方单元在一次运行中测量四条路径。组合决策使用实测中位数；服务时间主导项只作为再归因线索，不替代事件依赖与关键路径分析。


In [2]:
import json, platform
import torch
from tools.performance_optimization_runtime import run_candidate_composition_matrix
from tools.project_runtime import resolve_project_root

RESULTS_DIR = resolve_project_root() / RESULTS_RELATIVE_DIR
if RUN_MODE == "real_gpu":
    matrix = run_candidate_composition_matrix(
        hidden_size=GPU_CONFIG["hidden_size"], intermediate_size=GPU_CONFIG["intermediate_size"],
        token_counts=GPU_CONFIG["token_counts"], pipeline_batches=GPU_CONFIG["pipeline_batches"],
        warmup_steps=GPU_CONFIG["warmup_steps"], repeat_steps=GPU_CONFIG["repeat_steps"],
        seed=GPU_CONFIG["seed"],
        variants=["baseline_fp32_serial", "bf16_serial", "fp32_overlap", "bf16_overlap"],
    )
    if matrix.get("protocol", {}).get("measurement_order") != "rotating_order_within_each_round":
        raise RuntimeError("当前内核仍缓存旧版性能运行时；请重启内核后从 5.1 开始重新运行。")
    loads = {name: {item["token_count"]: item for item in data["loads"]} for name, data in matrix["variants"].items()}
    comparisons = []
    for token_count in GPU_CONFIG["token_counts"]:
        base = loads["baseline_fp32_serial"][token_count]
        singles = {name: loads[name][token_count] for name in ("bf16_serial", "fp32_overlap")}
        combo = loads["bf16_overlap"][token_count]
        best_single = min(singles, key=lambda name: singles[name]["latency"]["median_ms"])
        best_value = singles[best_single]["latency"]["median_ms"]
        base_value = base["latency"]["median_ms"]
        combo_value = combo["latency"]["median_ms"]
        gain_vs_baseline = (base_value - combo_value) / base_value
        gain_vs_best_single = (best_value - combo_value) / best_value
        quality_passed = combo["cosine_vs_fp32"] >= GPU_CONFIG["minimum_cosine"]
        combo_p95_ratio = combo["latency"]["p95_ms"] / singles[best_single]["latency"]["p95_ms"]
        tail_passed = combo_p95_ratio <= GPU_CONFIG["maximum_p95_ratio"]
        best_single_passed = best_value < base_value
        if not quality_passed or not tail_passed or gain_vs_baseline <= 0:
            decision = "keep_single" if best_single_passed else "rollback_baseline"
            rollback_to = best_single if best_single_passed else "baseline_fp32_serial"
        elif gain_vs_best_single >= GPU_CONFIG["minimum_marginal_gain"]:
            decision, rollback_to = "keep_combination", best_single
        else:
            decision, rollback_to = "keep_single", best_single
        services = combo["service_times"]
        dominant_service = "compute" if services["compute_median_ms"] >= services["h2d_median_ms"] else "h2d"
        comparisons.append({
            "token_count": token_count, "best_single": best_single,
            "gain_vs_baseline": gain_vs_baseline, "gain_vs_best_single": gain_vs_best_single,
            "combination_p95_ratio_vs_best_single": combo_p95_ratio, "tail_passed": tail_passed,
            "combination_cosine_vs_fp32": combo["cosine_vs_fp32"],
            "combination_service_times": services, "dominant_service_time": dominant_service,
            "decision": decision, "rollback_to": rollback_to,
        })
    record = {
        "schema_version": "performance-composition-mechanism/v1", "profile_id": PROFILE_ID,
        "semantic_id": SEMANTIC_ID, "run_label": RUN_LABEL,
        "framework": {"python": platform.python_version(), "torch": torch.__version__, "cuda": torch.version.cuda},
        "hardware": {"device": torch.cuda.get_device_name(0), "total_memory_bytes": torch.cuda.get_device_properties(0).total_memory},
        "workload": {"id": WORKLOAD_ID, "version": WORKLOAD_VERSION, **GPU_CONFIG},
        "matrix": matrix, "comparisons": comparisons, "failure": None,
        "evidence_level": "local_mechanism_benchmark", "decision": "review_per_load",
    }
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)
    result_path = RESULTS_DIR / f"{SEMANTIC_ID.lower()}_{RUN_LABEL}.json"
    result_path.write_text(json.dumps(record, ensure_ascii=False, indent=2), encoding="utf-8")
    print(json.dumps({"result_path": str(result_path), "comparisons": comparisons}, ensure_ascii=False, indent=2))
else:
    print({"run_mode": RUN_MODE, "profile_id": PROFILE_ID, "semantic_id": SEMANTIC_ID, "config": GPU_CONFIG})


{
  "result_path": "/data/llm-algo-leetcode-datawhale/llm-algo-leetcode/benchmarks/results/perf_composition/perf-composition_rtx5070ti_composition_bf16_overlap_v1.json",
  "comparisons": [
    {
      "token_count": 32,
      "best_single": "bf16_serial",
      "gain_vs_baseline": 0.6847164914816218,
      "gain_vs_best_single": 0.15127889469352995,
      "combination_cosine_vs_fp32": 0.9999672770500183,
      "combination_service_times": {
        "h2d_median_ms": 0.011888000182807446,
        "compute_median_ms": 0.18163200467824936
      },
      "dominant_service_time": "compute",
      "decision": "keep_combination",
      "rollback_to": "bf16_serial"
    },
    {
      "token_count": 256,
      "best_single": "bf16_serial",
      "gain_vs_baseline": 0.6588628285208328,
      "gain_vs_best_single": -0.012626295937905464,
      "combination_cosine_vs_fp32": 1.0000321865081787,
      "combination_service_times": {
        "h2d_median_ms": 0.07620799914002419,
        "compute_median

#### 5.3 读取结果

读取同一 `RUN_LABEL` 的结果文件，逐负载检查最佳单项、组合边际、输出质量、服务时间主导项和回退目标。


In [3]:
if RUN_MODE == "real_gpu":
    result_path = RESULTS_DIR / f"{SEMANTIC_ID.lower()}_{RUN_LABEL}.json"
    if not result_path.is_file():
        raise FileNotFoundError(f"未找到结果，请先运行 5.2：{result_path}")
    composition_record = json.loads(result_path.read_text(encoding="utf-8"))
    print(json.dumps(composition_record["comparisons"], ensure_ascii=False, indent=2))
else:
    composition_record = {"status": "not_run"}
    print(composition_record)


[
  {
    "token_count": 32,
    "best_single": "bf16_serial",
    "gain_vs_baseline": 0.6847164914816218,
    "gain_vs_best_single": 0.15127889469352995,
    "combination_cosine_vs_fp32": 0.9999672770500183,
    "combination_service_times": {
      "h2d_median_ms": 0.011888000182807446,
      "compute_median_ms": 0.18163200467824936
    },
    "dominant_service_time": "compute",
    "decision": "keep_combination",
    "rollback_to": "bf16_serial"
  },
  {
    "token_count": 256,
    "best_single": "bf16_serial",
    "gain_vs_baseline": 0.6588628285208328,
    "gain_vs_best_single": -0.012626295937905464,
    "combination_cosine_vs_fp32": 1.0000321865081787,
    "combination_service_times": {
      "h2d_median_ms": 0.07620799914002419,
      "compute_median_ms": 0.5403200089931488
    },
    "dominant_service_time": "compute",
    "decision": "keep_single",
    "rollback_to": "bf16_serial"
  }
]


#### 5.4 解释证据与决策

RTX 5070 Ti 的 v2 实测采用 24 轮配对样本，并在每轮轮换四条路径的执行顺序。BF16 是两档最佳单项；BF16 与重叠组合在两档负载均保留正边际，且尾延迟没有回归。

| 观察项 | 读取字段 | v2 实测结果 | 解释与动作 |
| --- | --- | --- | --- |
| 运行与质量 | `failure`、协议、组合余弦相似度 | 无失败；24 轮配对测量；两档质量通过 | 当前结果可用于组合决策 |
| 单项基线 | 两个单项中位数 | 两档最佳单项均为 BF16 串行 | BF16 是组合失败时的回退目标 |
| 组合边际 | `gain_vs_best_single`、同轮样本 | 32/256 Token 分别约 +5.6%/+12.2%；两档均为 24/24 轮正边际 | 当前两档负载保留 BF16 与重叠组合 |
| 尾延迟 | 组合相对最佳单项的 P95 比值 | 约 0.73/0.89 | 两档均通过不高于 1.05 的尾延迟门槛 |
| 归因与边界 | H2D 与计算服务时间 | 两档均由计算服务时间主导 | 结论只适用于当前矩阵；输入形状或流水线改变后必须复测 |
| 最终动作 | `decision`、`rollback_to` | 两档均 `keep_combination` | 接受当前组合，回退路径固定为 BF16 串行 |


## 相关阅读

- [性能优化 · 07 端到端性能工程与决策](../topic_discussion/performance_optimization/07_end_to_end_performance_engineering.md)
- [PERF-CANDIDATE：从约束到候选与代价转移](./PERF-CANDIDATE_Constraint_to_Candidate_and_Cost_Transfer.ipynb)
- [PERF-TRACE：执行轨迹依赖与关键路径](./PERF-TRACE_Trace_Dependencies_and_Critical_Path.ipynb)
- [PERF-REGRESSION：配对基准与性能回归阈值](./PERF-REGRESSION_Paired_Benchmark_and_Regression_Thresholds.ipynb)
